# PanDx Reproduction Notebook
**AI-assisted Early Detection of Pancreatic Ductal Adenocarcinoma on Contrast-enhanced CT**

Liu et al. (2025) — 1st place, PANORAMA Challenge
- Paper: https://arxiv.org/abs/2503.10068
- Code:  https://github.com/han-liu/PanDx

## Pipeline overview
```
CT scan (full resolution)
   │
   ▼
[Dataset Prep]  Data exploration → DASE split → nnU-Net format conversion
   │
   ▼
[Stage 1]  Downsample → nnU-Net (Dataset103) → Pancreas segmentation mask
   │
   ▼
[Stage 2]  Crop ROI (±100/50/15 mm) → nnU-Net (Dataset107, CE loss) → PDAC prob map
   │
   ▼
[Post]     Peak-scaled candidate extraction (α = 1/15) → patient likelihood score
```

## PANORAMA label map
| Label | Structure |
|---|---|
| 0 | Background |
| 1 | Pancreas parenchyma |
| 2 | PDAC lesion |
| 3 | Pancreatic duct |
| 4 | Common bile duct |
| 5 | Veins |
| 6 | Arteries |

## 0. Environment setup & imports

In [ ]:
import os
import os.path as osp
import json
import shutil
import subprocess
import hashlib
import time
import warnings
from glob import glob
from pathlib import Path

import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm import tqdm

warnings.filterwarnings('ignore')

# Add local packages to path
import sys
REPO_ROOT = osp.abspath('.')          # /Desktop/PanDx
sys.path.insert(0, osp.join(REPO_ROOT, 'packages', 'nnunetv2'))
sys.path.insert(0, osp.join(REPO_ROOT, 'packages', 'report-guided-annotation', 'src'))

from report_guided_annotation.extract_lesion_candidates import extract_lesion_candidates

print('Imports OK')
print('REPO_ROOT:', REPO_ROOT)

## 1. Configuration — set your paths here

In [ ]:
# ── Raw PANORAMA dataset ───────────────────────────────────────────────────────
# Folder containing all CT scans (.mha or .nii.gz)
IMAGE_DIR = osp.join(REPO_ROOT, 'workspace', 'raw_data', 'images')
# Folder containing corresponding segmentation masks (same filenames)
LABEL_DIR = osp.join(REPO_ROOT, 'workspace', 'raw_data', 'labels')

# ── Inference I/O paths ───────────────────────────────────────────────────────
# For single-case testing use the test_example folder;
# for the full dataset point INPUT_DIR at IMAGE_DIR above
INPUT_DIR   = osp.join(REPO_ROOT, 'workspace', 'test_example', 'input')
OUTPUT_DIR  = osp.join(REPO_ROOT, 'workspace', 'test_example', 'output')
MODEL_DIR   = osp.join(REPO_ROOT, 'workspace', 'nnUNet_results')
WORKING_DIR = osp.join(OUTPUT_DIR, 'itm')

# ── nnU-Net dataset paths ─────────────────────────────────────────────────────
NNUNET_RAW          = osp.join(REPO_ROOT, 'workspace', 'nnUNet_raw')
NNUNET_PREPROCESSED = osp.join(REPO_ROOT, 'workspace', 'nnUNet_preprocessed')
DATASET_NAME        = 'Dataset107_PDAC_Detection'
DATASET_ID          = 107

# ── Model identifiers ─────────────────────────────────────────────────────────
STAGE1_TASK    = 103
STAGE1_TRAINER = 'nnUNetTrainer'
STAGE1_PLAN    = 'nnUNetPlans'

STAGE2_TASK    = 107
STAGE2_TRAINER = 'nnUNetTrainerCELossLesionSplit'
STAGE2_PLAN    = 'nnUNetPlans_v3'

# ── Hyper-parameters ──────────────────────────────────────────────────────────
DOWNSAMPLE_SPACING = (4.5, 4.5, 9.0)   # mm — low-res spacing for Stage 1
ROI_MARGINS        = [100, 50, 15]      # mm — x/y/z padding around pancreas bbox
INV_ALPHA          = 15                 # τ = (1/15) · P(x*)  peak-scaling factor
RANDOM_SEED        = 42                 # fixed seed for DASE split reproducibility
N_FOLDS            = 5

for d in [OUTPUT_DIR, WORKING_DIR, osp.join(OUTPUT_DIR, 'pdac-detection-map'),
          NNUNET_RAW, NNUNET_PREPROCESSED]:
    os.makedirs(d, exist_ok=True)

print('Config ready.')
print(f'  IMAGE_DIR : {IMAGE_DIR}')
print(f'  LABEL_DIR : {LABEL_DIR}')
print(f'  OUTPUT_DIR: {OUTPUT_DIR}')
print(f'  MODEL_DIR : {MODEL_DIR}')

## 2. Helper functions

In [ ]:
def get_file_extension(path: str) -> str:
    base, ext = osp.splitext(path)
    if ext == '.gz' and base.endswith('.nii'):
        return '.nii.gz'
    return ext


def resample_img(itk_image, out_spacing=(2.0, 2.0, 2.0), is_label=False,
                 out_size=None, out_origin=None, out_direction=None):
    """Resample an ITK image to the requested voxel spacing."""
    orig_spacing = itk_image.GetSpacing()
    orig_size    = itk_image.GetSize()
    if out_size is None:
        out_size = [
            int(np.round(orig_size[i] * orig_spacing[i] / out_spacing[i]))
            for i in range(3)
        ]
    rs = sitk.ResampleImageFilter()
    rs.SetOutputSpacing(out_spacing)
    rs.SetSize(out_size)
    rs.SetOutputDirection(out_direction or itk_image.GetDirection())
    rs.SetOutputOrigin(out_origin or itk_image.GetOrigin())
    rs.SetTransform(sitk.Transform())
    rs.SetDefaultPixelValue(itk_image.GetPixelIDValue())
    rs.SetInterpolator(sitk.sitkNearestNeighbor if is_label else sitk.sitkBSpline)
    return rs.Execute(itk_image)


def downsample_dataset(img_dir: str, save_dir: str,
                       spacing=DOWNSAMPLE_SPACING) -> None:
    """Downsample all images in img_dir to low-res spacing for Stage-1 inference."""
    os.makedirs(save_dir, exist_ok=True)
    img_paths = sorted(glob(osp.join(img_dir, '*.*')))
    assert img_paths, f'No images found in {img_dir}'
    for p in tqdm(img_paths, desc='Downsample'):
        ext  = get_file_extension(p)
        img  = sitk.ReadImage(p, sitk.sitkFloat32)
        resampled = resample_img(img, spacing)
        out_name  = osp.basename(p).replace(ext, '_0000.nii.gz')
        sitk.WriteImage(resampled, osp.join(save_dir, out_name))


def crop_roi(img_dir: str, low_mask_dir: str, save_dir: str,
             margins=ROI_MARGINS) -> dict:
    """
    Crop the high-resolution CT around the predicted pancreas bounding box.
    Returns a dict mapping case-id → crop coordinate slices.
    """
    os.makedirs(save_dir, exist_ok=True)
    img_paths = sorted(glob(osp.join(img_dir, '*.*')))
    crop_coords = {}
    for p in tqdm(img_paths, desc='Crop ROI'):
        ext       = get_file_extension(p)
        mask_path = osp.join(low_mask_dir, osp.basename(p).replace(ext, '.nii.gz'))
        img       = sitk.ReadImage(p, sitk.sitkFloat32)
        low_mask  = sitk.ReadImage(mask_path)
        mask_np   = sitk.GetArrayFromImage(low_mask)
        mask_np   = (mask_np == 1).astype(np.uint8)
        nz = np.nonzero(mask_np)
        min_x, max_x = int(nz[2].min()), int(nz[2].max())
        min_y, max_y = int(nz[1].min()), int(nz[1].max())
        min_z, max_z = int(nz[0].min()), int(nz[0].max())
        sp = low_mask.TransformIndexToPhysicalPoint
        ip = img.TransformPhysicalPointToIndex
        start_idx  = ip(sp((min_x, min_y, min_z)))
        finish_idx = ip(sp((max_x, max_y, max_z)))
        spacing = img.GetSpacing()
        size    = img.GetSize()
        mx = int(margins[0] / spacing[0])
        my = int(margins[1] / spacing[1])
        mz = int(margins[2] / spacing[2])
        xs = max(0, start_idx[0] - mx);  xf = min(size[0], finish_idx[0] + mx)
        ys = max(0, start_idx[1] - my);  yf = min(size[1], finish_idx[1] + my)
        zs = max(0, start_idx[2] - mz);  zf = min(size[2], finish_idx[2] + mz)
        cropped = img[xs:xf, ys:yf, zs:zf]
        case_id = osp.basename(p).replace(ext, '')
        crop_coords[case_id] = dict(x_start=xs, x_finish=xf,
                                    y_start=ys, y_finish=yf,
                                    z_start=zs, z_finish=zf)
        sitk.WriteImage(cropped,
                        osp.join(save_dir, osp.basename(p).replace(ext, '_0000.nii.gz')))
    return crop_coords


def run_nnunet_predict(model_dir: str, input_dir: str, output_dir: str,
                       task: int, trainer: str = 'nnUNetTrainer',
                       plan: str = 'nnUNetPlans',
                       configuration: str = '3d_fullres',
                       checkpoint: str = 'checkpoint_final.pth',
                       folds: str = '0,1,2,3,4',
                       save_probs: bool = True,
                       tta: bool = True) -> None:
    """Call nnUNetv2_predict as a subprocess."""
    os.environ['RESULTS_FOLDER'] = model_dir
    os.makedirs(output_dir, exist_ok=True)
    cmd = [
        'nnUNetv2_predict',
        '-d',  str(task),
        '-i',  input_dir,
        '-o',  output_dir,
        '-c',  configuration,
        '-tr', trainer,
        '-p',  plan,
        '--continue_prediction',
        '-f',  *folds.split(','),
        '-chk', checkpoint,
    ]
    if save_probs:
        cmd.append('--save_probabilities')
    if not tta:
        cmd.append('--disable_tta')
    print('Running:', ' '.join(str(c) for c in cmd))
    subprocess.check_call(cmd)


def postprocess_probmap(npz_path: str) -> np.ndarray:
    """Load .npz from nnU-Net and extract the PDAC (class 1) probability map."""
    data = np.load(npz_path)
    return data['probabilities'][1].astype(np.float32)


def build_full_size_detection_map(prob_map: np.ndarray,
                                  crop_coords: dict,
                                  reference_image: sitk.Image,
                                  inv_alpha: int = INV_ALPHA):
    """
    Apply peak-scaled lesion candidate extraction (τ = prob_map.max() / inv_alpha),
    then paste the result back into a full-size volume.
    Returns (detection_map_itk, patient_likelihood_score).
    """
    lesion_candidates, _, _ = extract_lesion_candidates(
        prob_map, dynamic_threshold_factor=inv_alpha
    )
    patient_score = float(np.max(lesion_candidates))
    full_shape = sitk.GetArrayFromImage(reference_image).shape
    full_map   = np.zeros(full_shape, dtype=np.float32)
    full_map[
        crop_coords['z_start']:crop_coords['z_finish'],
        crop_coords['y_start']:crop_coords['y_finish'],
        crop_coords['x_start']:crop_coords['x_finish'],
    ] = lesion_candidates
    out_itk = sitk.GetImageFromArray(full_map)
    out_itk.CopyInformation(reference_image)
    return out_itk, patient_score


def write_json(path: str, content: dict) -> None:
    with open(path, 'w') as f:
        json.dump(content, f, indent=4)


print('Helper functions defined.')

---
# Part A — Dataset Preparation
---

## 3. Dataset inspection
Summarise what images are in `IMAGE_DIR` and display one representative slice.

In [ ]:
img_paths = sorted([
    p for p in glob(osp.join(IMAGE_DIR, '*.*'))
    if not p.endswith('.json') and not osp.isdir(p)
])
print(f'Found {len(img_paths)} image(s) in {IMAGE_DIR}:')
for p in img_paths[:10]:   # show first 10 to avoid flooding output
    img = sitk.ReadImage(p)
    print(f'  {osp.basename(p):45s}  size={img.GetSize()}  '
          f'spacing={tuple(f"{s:.2f}" for s in img.GetSpacing())}')
if len(img_paths) > 10:
    print(f'  ... and {len(img_paths) - 10} more')

In [ ]:
# Visualise a central axial slice of the first image
if img_paths:
    ex    = sitk.ReadImage(img_paths[0], sitk.sitkFloat32)
    arr   = sitk.GetArrayFromImage(ex)
    mid_z = arr.shape[0] // 2
    plt.figure(figsize=(6, 6))
    plt.imshow(arr[mid_z], cmap='gray', vmin=-200, vmax=300)
    plt.title(f'{osp.basename(img_paths[0])} — axial slice {mid_z}')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

## 4. Data exploration

Compute for every case:
- Whether it is PDAC-positive (label 2 present in mask)
- PDAC lesion volume in mm³
- Age and sex from the clinical metadata JSON (if available)

**Expected finding (Fig. 1 of the paper):** lesion size is heavily right-skewed — most lesions are small (< 1 000 mm³).

In [ ]:
label_paths = sorted([
    p for p in glob(osp.join(LABEL_DIR, '*.*'))
    if not p.endswith('.json') and not osp.isdir(p)
])

cases = []
for label_path in tqdm(label_paths, desc='Scanning labels'):
    ext     = get_file_extension(label_path)
    case_id = osp.basename(label_path).replace(ext, '')

    mask    = sitk.ReadImage(label_path)
    arr     = sitk.GetArrayFromImage(mask)
    spacing = mask.GetSpacing()                        # (x, y, z) in mm
    vox_vol = spacing[0] * spacing[1] * spacing[2]    # mm³ per voxel

    pdac_voxels  = int((arr == 2).sum())
    pdac_vol_mm3 = pdac_voxels * vox_vol

    entry = {
        'case_id':      case_id,
        'is_pdac':      pdac_voxels > 0,
        'pdac_vol_mm3': pdac_vol_mm3,
        'age':          None,
        'sex':          None,
    }

    # Load clinical metadata JSON (one per case, same stem as image)
    for json_dir in [IMAGE_DIR, LABEL_DIR]:
        json_path = osp.join(json_dir, f'{case_id}.json')
        if osp.exists(json_path):
            with open(json_path) as f:
                meta = json.load(f)
            entry['age'] = meta.get('age')
            entry['sex'] = meta.get('sex')
            break

    cases.append(entry)

df = pd.DataFrame(cases)
print(f'Total cases   : {len(df)}')
print(f'PDAC positive : {df.is_pdac.sum()}')
print(f'PDAC negative : {(~df.is_pdac).sum()}')
print(f'\nLesion volume stats (PDAC-positive cases only):')
print(df[df.is_pdac]['pdac_vol_mm3'].describe().round(1))

In [ ]:
# ── Lesion size distribution ──────────────────────────────────────────────────
pdac_vols = df[df.is_pdac]['pdac_vol_mm3']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(pdac_vols, bins=50, color='tomato', edgecolor='white')
axes[0].set_xlabel('Lesion volume (mm³)')
axes[0].set_ylabel('Number of cases')
axes[0].set_title('Lesion size distribution — linear scale\n(note strong right skew toward small lesions)')

axes[1].hist(np.log1p(pdac_vols), bins=50, color='steelblue', edgecolor='white')
axes[1].set_xlabel('log(1 + lesion volume)')
axes[1].set_ylabel('Number of cases')
axes[1].set_title('Lesion size distribution — log scale')

# Mark quartile boundaries on the log-scale plot
for q, c in zip([0.25, 0.50, 0.75], ['gold', 'orange', 'red']):
    val = np.log1p(pdac_vols.quantile(q))
    axes[1].axvline(val, color=c, linestyle='--', label=f'Q{int(q*4)} boundary')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ── Age / sex distribution ────────────────────────────────────────────────────
has_meta = df['age'].notna().any()

if has_meta:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Age histogram split by PDAC status
    for label, colour in [('PDAC+', 'tomato'), ('PDAC-', 'steelblue')]:
        is_p = label == 'PDAC+'
        ages = df[df.is_pdac == is_p]['age'].dropna()
        axes[0].hist(ages, bins=20, alpha=0.6, color=colour, label=label, edgecolor='white')
    axes[0].set_xlabel('Age (years)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Age distribution')
    axes[0].legend()

    # Sex breakdown
    sex_counts = df.groupby(['sex', 'is_pdac']).size().unstack(fill_value=0)
    sex_counts.columns = ['PDAC-', 'PDAC+']
    sex_counts.plot(kind='bar', ax=axes[1], color=['steelblue', 'tomato'],
                    edgecolor='white', rot=0)
    axes[1].set_title('Sex distribution')
    axes[1].set_ylabel('Count')

    plt.tight_layout()
    plt.show()
else:
    print('No clinical metadata found — skipping age/sex plots.')
    print('(Expected JSON files alongside images with "age" and "sex" keys.)')

## 5. DASE split — Distribution-Aware Stratified Evaluation

**Goal:** ensure every fold gets a proportional share of both PDAC+/- cases *and* all lesion size ranges.

**Algorithm:**
1. Sort PDAC+ cases by lesion volume → divide into 4 quartile bins (Q1–Q4)
2. Within each bin, assign cases round-robin across 5 folds
3. Distribute PDAC- cases round-robin to preserve the global PDAC+/- ratio
4. Fix `RANDOM_SEED = 42` for full reproducibility

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

pdac_pos = df[df.is_pdac].copy().sort_values('pdac_vol_mm3').reset_index(drop=True)
pdac_neg = df[~df.is_pdac].copy().reset_index(drop=True)

# Assign quartile size bin to each PDAC+ case
pdac_pos['size_bin'] = pd.qcut(
    pdac_pos['pdac_vol_mm3'],
    q=4,
    labels=['Q1 (0-25%)', 'Q2 (25-50%)', 'Q3 (50-75%)', 'Q4 (75-100%)']
)

print('PDAC+ cases per size quartile:')
print(pdac_pos['size_bin'].value_counts().sort_index().to_string())
print()

# Round-robin assignment within each quartile
fold_assignments = {}
for bin_label in ['Q1 (0-25%)', 'Q2 (25-50%)', 'Q3 (50-75%)', 'Q4 (75-100%)']:
    bin_cases = pdac_pos[pdac_pos.size_bin == bin_label]['case_id'].tolist()
    rng.shuffle(bin_cases)
    for i, case_id in enumerate(bin_cases):
        fold_assignments[case_id] = i % N_FOLDS

# Round-robin assignment for PDAC- cases
neg_cases = pdac_neg['case_id'].tolist()
rng.shuffle(neg_cases)
for i, case_id in enumerate(neg_cases):
    fold_assignments[case_id] = i % N_FOLDS

df['fold'] = df['case_id'].map(fold_assignments)

# ── Summary table ─────────────────────────────────────────────────────────────
print('Fold composition:')
for fold in range(N_FOLDS):
    fold_df = df[df.fold == fold]
    n_pos   = fold_df.is_pdac.sum()
    n_neg   = (~fold_df.is_pdac).sum()
    ratio   = n_pos / len(fold_df) * 100
    print(f'  Fold {fold}: {len(fold_df):4d} cases | '
          f'PDAC+ = {n_pos:3d}  PDAC- = {n_neg:3d}  ({ratio:.1f}% positive)')

print()
print('PDAC+ lesion-size distribution per fold:')
print(
    pdac_pos
    .assign(fold=pdac_pos.case_id.map(fold_assignments))
    .groupby(['fold', 'size_bin'], observed=True)
    .size()
    .unstack(fill_value=0)
    .to_string()
)

In [ ]:
# ── Visualise the DASE split ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Stacked bar: PDAC+ (by quartile) and PDAC- counts per fold
fold_pos = (
    pdac_pos
    .assign(fold=pdac_pos.case_id.map(fold_assignments))
    .groupby(['fold', 'size_bin'], observed=True)
    .size()
    .unstack(fill_value=0)
)
fold_neg = df[~df.is_pdac].groupby('fold').size().rename('PDAC-')

colours = ['#4CAF50', '#2196F3', '#FF9800', '#F44336']
bottom  = np.zeros(N_FOLDS)
for col, colour in zip(fold_pos.columns, colours):
    axes[0].bar(fold_pos.index, fold_pos[col], bottom=bottom,
                color=colour, label=str(col), edgecolor='white')
    bottom += fold_pos[col].values
axes[0].bar(fold_neg.index, fold_neg.values, bottom=bottom,
            color='#9E9E9E', label='PDAC-', edgecolor='white')
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('Number of cases')
axes[0].set_title('DASE split — cases per fold by size quartile')
axes[0].legend(fontsize=8, loc='upper right')

# PDAC+ ratio per fold
ratio_per_fold = [df[df.fold == f].is_pdac.mean() * 100 for f in range(N_FOLDS)]
global_ratio   = df.is_pdac.mean() * 100
axes[1].bar(range(N_FOLDS), ratio_per_fold, color='tomato', edgecolor='white')
axes[1].axhline(global_ratio, color='black', linestyle='--', label=f'Global ratio ({global_ratio:.1f}%)')
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('% PDAC-positive')
axes[1].set_title('PDAC+ ratio per fold (should be uniform)')
axes[1].set_ylim(0, 100)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

# Save the split to CSV
split_csv = osp.join(REPO_ROOT, 'workspace', 'dase_split.csv')
df[['case_id', 'is_pdac', 'pdac_vol_mm3', 'fold']].to_csv(split_csv, index=False)
print(f'\nSplit saved to: {split_csv}')

## 6. Reproducibility check

Re-run this cell from scratch at any time.  
The **MD5 checksum must be identical** every run — if it changes you have a non-determinism bug.

In [ ]:
df_check  = pd.read_csv(split_csv)
fold_str  = df_check.sort_values('case_id')['fold'].astype(str).str.cat()
checksum  = hashlib.md5(fold_str.encode()).hexdigest()

print('=' * 50)
print(f'  Split checksum (MD5): {checksum}')
print('=' * 50)
print('Save this value. Re-running with RANDOM_SEED=42')
print('must always produce the same checksum.')
print()
print('Fold sizes:')
print(df_check.groupby('fold').size().rename('n_cases').to_string())

## 7. Convert dataset to nnU-Net format

nnU-Net requires:
```
nnUNet_raw/Dataset107_PDAC_Detection/
    dataset.json
    imagesTr/   ← images named  <case_id>_0000.nii.gz
    labelsTr/   ← labels named  <case_id>.nii.gz
```
Labels are kept as-is (multi-class 0–6) since nnU-Net accepts them directly.

In [ ]:
out_images = osp.join(NNUNET_RAW, DATASET_NAME, 'imagesTr')
out_labels = osp.join(NNUNET_RAW, DATASET_NAME, 'labelsTr')
os.makedirs(out_images, exist_ok=True)
os.makedirs(out_labels, exist_ok=True)

skipped = 0
for _, row in tqdm(df.iterrows(), total=len(df), desc='Copying files'):
    cid = row['case_id']

    # Locate source image (try .nii.gz and .mha)
    src_img = None
    for ext in ['.nii.gz', '.mha', '.nii']:
        candidate = osp.join(IMAGE_DIR, f'{cid}{ext}')
        if osp.exists(candidate):
            src_img = candidate
            break
    if src_img is None:
        print(f'  WARNING: image not found for {cid} — skipping')
        skipped += 1
        continue

    # Locate source label
    src_lbl = None
    for ext in ['.nii.gz', '.mha', '.nii']:
        candidate = osp.join(LABEL_DIR, f'{cid}{ext}')
        if osp.exists(candidate):
            src_lbl = candidate
            break
    if src_lbl is None:
        print(f'  WARNING: label not found for {cid} — skipping')
        skipped += 1
        continue

    dst_img = osp.join(out_images, f'{cid}_0000.nii.gz')
    dst_lbl = osp.join(out_labels, f'{cid}.nii.gz')

    if not osp.exists(dst_img):
        # Convert .mha → .nii.gz if needed, otherwise plain copy
        if src_img.endswith('.nii.gz'):
            shutil.copy2(src_img, dst_img)
        else:
            img_itk = sitk.ReadImage(src_img, sitk.sitkFloat32)
            sitk.WriteImage(img_itk, dst_img)

    if not osp.exists(dst_lbl):
        if src_lbl.endswith('.nii.gz'):
            shutil.copy2(src_lbl, dst_lbl)
        else:
            lbl_itk = sitk.ReadImage(src_lbl)
            sitk.WriteImage(lbl_itk, dst_lbl)

n_copied = len(df) - skipped
print(f'\nCopied {n_copied} cases ({skipped} skipped).')

# Write dataset.json
dataset_json = {
    "channel_names": {"0": "CT"},
    "labels": {
        "background":          0,
        "pancreas":            1,
        "PDAC":                2,
        "pancreatic_duct":     3,
        "common_bile_duct":    4,
        "veins":               5,
        "arteries":            6
    },
    "numTraining": n_copied,
    "file_ending": ".nii.gz"
}
write_json(osp.join(NNUNET_RAW, DATASET_NAME, 'dataset.json'), dataset_json)
print(f'dataset.json written to {osp.join(NNUNET_RAW, DATASET_NAME)}')

## 8. nnU-Net plan & preprocess

This runs `nnUNetv2_plan_and_preprocess` **inside the notebook**.  
nnU-Net automatically handles: resampling, normalization, patch sizing, and architecture depth.

> **Tip:** If this cell takes > 10 minutes or the kernel crashes, run it in a terminal instead (copy the shell command printed below and paste it into your terminal).

In [ ]:
# Set required nnU-Net environment variables for this session
os.environ['nnUNet_raw']          = NNUNET_RAW
os.environ['nnUNet_preprocessed'] = NNUNET_PREPROCESSED
os.environ['nnUNet_results']      = MODEL_DIR

cmd = (
    f'nnUNetv2_plan_and_preprocess '
    f'-d {DATASET_ID} '
    f'--verify_dataset_integrity'
)
print('Command to run:')
print(f'  {cmd}')
print()
print('If you prefer the terminal, copy the command above and run it there.')
print('Otherwise, set RUN_PREPROCESS = True below and re-run this cell.')

In [ ]:
# Set to True only when ready — this step can take 30-60+ minutes on the full dataset
RUN_PREPROCESS = False

if RUN_PREPROCESS:
    print('Running nnU-Net plan & preprocess...')
    subprocess.check_call(cmd, shell=True)
    print('Done. Preprocessed files are in:', NNUNET_PREPROCESSED)
else:
    print('Skipped. Set RUN_PREPROCESS = True to run.')
    print('Or run in terminal:')
    print(f'  export nnUNet_raw="{NNUNET_RAW}"')
    print(f'  export nnUNet_preprocessed="{NNUNET_PREPROCESSED}"')
    print(f'  export nnUNet_results="{MODEL_DIR}"')
    print(f'  {cmd}')

---
# Part B — Inference Pipeline
---

## 9. Stage 1 — Pancreas localisation at low resolution

### 9a. Downsample to (4.5, 4.5, 9.0) mm

In [ ]:
LOW_IMAGE_DIR = osp.join(WORKING_DIR, 'LowImagesTr')
downsample_dataset(INPUT_DIR, LOW_IMAGE_DIR, DOWNSAMPLE_SPACING)
print('Downsampled images:', sorted(os.listdir(LOW_IMAGE_DIR)))

### 9b. nnU-Net inference — Dataset 103 (5-fold ensemble)

In [ ]:
LOW_PRED_DIR = osp.join(WORKING_DIR, 'LowPred')

run_nnunet_predict(
    model_dir  = MODEL_DIR,
    input_dir  = LOW_IMAGE_DIR,
    output_dir = LOW_PRED_DIR,
    task       = STAGE1_TASK,
    trainer    = STAGE1_TRAINER,
    plan       = STAGE1_PLAN,
    folds      = '0,1,2,3,4',
    save_probs = True,
    tta        = True,
)
print('Stage-1 predictions:', sorted(os.listdir(LOW_PRED_DIR)))

### 9c. Visualise Stage-1 segmentation overlay

In [ ]:
low_preds = sorted(glob(osp.join(LOW_PRED_DIR, '*.nii.gz')))
if low_preds:
    low_img_path  = sorted(glob(osp.join(LOW_IMAGE_DIR, '*.nii.gz')))[0]
    low_mask_path = low_preds[0]
    low_arr  = sitk.GetArrayFromImage(sitk.ReadImage(low_img_path, sitk.sitkFloat32))
    mask_arr = sitk.GetArrayFromImage(sitk.ReadImage(low_mask_path))
    best_z   = int(np.argmax((mask_arr == 1).sum(axis=(1, 2))))

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(low_arr[best_z], cmap='gray', vmin=-200, vmax=300)
    axes[0].set_title(f'Low-res CT — slice {best_z}')
    axes[0].axis('off')

    axes[1].imshow(low_arr[best_z],         cmap='gray',   vmin=-200, vmax=300)
    axes[1].imshow(mask_arr[best_z] == 1,   cmap='Reds',   alpha=0.4)
    axes[1].imshow(mask_arr[best_z] == 2,   cmap='Greens', alpha=0.5)
    axes[1].imshow(mask_arr[best_z] == 3,   cmap='Blues',  alpha=0.4)
    axes[1].set_title('Stage-1 segmentation overlay')
    axes[1].axis('off')

    patches = [
        mpatches.Patch(color='red',   alpha=0.6, label='Pancreas (1)'),
        mpatches.Patch(color='green', alpha=0.6, label='PDAC (2)'),
        mpatches.Patch(color='blue',  alpha=0.6, label='Duct (3)'),
    ]
    axes[1].legend(handles=patches, loc='lower right', fontsize=8)
    plt.tight_layout()
    plt.show()

## 10. Stage 1 → Stage 2: Crop high-resolution ROI

Expand the predicted pancreas bounding box by **100 × 50 × 15 mm³** in x/y/z.

In [ ]:
CROPPED_IMAGE_DIR = osp.join(WORKING_DIR, 'CroppedImages')

crop_coordinates = crop_roi(
    img_dir      = INPUT_DIR,
    low_mask_dir = LOW_PRED_DIR,
    save_dir     = CROPPED_IMAGE_DIR,
    margins      = ROI_MARGINS,
)

print('\nCrop coordinates:')
for case_id, coords in crop_coordinates.items():
    print(f'  {case_id}: {coords}')

In [ ]:
# Visualise one cropped ROI vs the full scan
infer_img_paths = sorted([
    p for p in glob(osp.join(INPUT_DIR, '*.*'))
    if not p.endswith('.json') and not osp.isdir(p)
])
if infer_img_paths:
    full_arr    = sitk.GetArrayFromImage(sitk.ReadImage(infer_img_paths[0], sitk.sitkFloat32))
    cropped_arr = sitk.GetArrayFromImage(
        sitk.ReadImage(sorted(glob(osp.join(CROPPED_IMAGE_DIR, '*.nii.gz')))[0],
                       sitk.sitkFloat32)
    )
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(full_arr[full_arr.shape[0]//2],       cmap='gray', vmin=-200, vmax=300)
    axes[0].set_title('Full CT (mid-axial)')
    axes[0].axis('off')
    axes[1].imshow(cropped_arr[cropped_arr.shape[0]//2], cmap='gray', vmin=-200, vmax=300)
    axes[1].set_title('Cropped ROI (mid-axial)')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

## 11. Stage 2 — Fine-scale PDAC detection

nnU-Net with **ResU-Net backbone** and **CE-only loss** (`nnUNetTrainerCELossLesionSplit`).

We run each fold **separately first** (needed for Phase 3 uncertainty quantification),
then run the full 5-fold ensemble for final predictions.

In [ ]:
CROPPED_PRED_DIR = osp.join(WORKING_DIR, 'CroppedPred')

# ── Step A: run each fold separately (saves per-fold .npz for uncertainty) ────
PER_FOLD_PRED_DIRS = {}
for fold in range(N_FOLDS):
    fold_out = osp.join(WORKING_DIR, f'CroppedPred_fold{fold}')
    PER_FOLD_PRED_DIRS[fold] = fold_out
    run_nnunet_predict(
        model_dir  = MODEL_DIR,
        input_dir  = CROPPED_IMAGE_DIR,
        output_dir = fold_out,
        task       = STAGE2_TASK,
        trainer    = STAGE2_TRAINER,
        plan       = STAGE2_PLAN,
        folds      = str(fold),
        save_probs = True,
        tta        = True,
    )

# ── Step B: run full 5-fold ensemble for final predictions ────────────────────
run_nnunet_predict(
    model_dir  = MODEL_DIR,
    input_dir  = CROPPED_IMAGE_DIR,
    output_dir = CROPPED_PRED_DIR,
    task       = STAGE2_TASK,
    trainer    = STAGE2_TRAINER,
    plan       = STAGE2_PLAN,
    folds      = '0,1,2,3,4',
    save_probs = True,
    tta        = True,
)
print('Stage-2 ensemble predictions:', sorted(os.listdir(CROPPED_PRED_DIR)))

## 12. Post-processing — peak-scaled lesion candidate extraction

**Algorithm (from the paper):**
1. Set threshold τ = (1 / `inv_alpha`) × P(x*) where x* is the max-probability voxel.
2. Keep connected components above τ → lesion candidates.
3. Patient-level likelihood = max voxel probability in the candidate map.
4. Paste back into a full-size volume aligned to the original CT.

In [ ]:
npz_fps = sorted(glob(osp.join(CROPPED_PRED_DIR, '*.npz')))
infer_img_paths = sorted([
    p for p in glob(osp.join(INPUT_DIR, '*.*'))
    if not p.endswith('.json') and not osp.isdir(p)
])

assert len(npz_fps) == len(infer_img_paths), (
    f'Mismatch: {len(npz_fps)} predictions vs {len(infer_img_paths)} input images'
)

likelihoods = {}
for npz_fp, img_fp in zip(npz_fps, infer_img_paths):
    ext     = get_file_extension(img_fp)
    case_id = osp.basename(npz_fp)[:-4]
    assert osp.basename(img_fp).replace(ext, '') == case_id, \
        f'File order mismatch: {osp.basename(img_fp)} vs {osp.basename(npz_fp)}'

    ref_img  = sitk.ReadImage(img_fp, sitk.sitkFloat32)
    prob_map = postprocess_probmap(npz_fp)
    det_map, score = build_full_size_detection_map(
        prob_map, crop_coordinates[case_id], ref_img, INV_ALPHA
    )
    out_path = osp.join(OUTPUT_DIR, 'pdac-detection-map', f'{case_id}.nii.gz')
    sitk.WriteImage(det_map, out_path)
    likelihoods[case_id] = score
    print(f'  {case_id:40s}  likelihood = {score:.4f}')

write_json(osp.join(OUTPUT_DIR, 'pdac-likelihood.json'), likelihoods)
print(f'\nLikelihood scores → {osp.join(OUTPUT_DIR, "pdac-likelihood.json")}')

## 13. Visualise detection maps

In [ ]:
det_map_paths = sorted(glob(osp.join(OUTPUT_DIR, 'pdac-detection-map', '*.nii.gz')))

for det_path, img_fp in zip(det_map_paths, infer_img_paths):
    ct_arr  = sitk.GetArrayFromImage(sitk.ReadImage(img_fp,   sitk.sitkFloat32))
    det_arr = sitk.GetArrayFromImage(sitk.ReadImage(det_path, sitk.sitkFloat32))
    case_id = osp.basename(det_path).replace('.nii.gz', '')
    best_z  = int(np.argmax(det_arr.max(axis=(1, 2))))

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(ct_arr[best_z],  cmap='gray', vmin=-200, vmax=300)
    axes[0].set_title('CT (axial)')
    axes[0].axis('off')
    axes[1].imshow(det_arr[best_z], cmap='hot',  vmin=0,    vmax=1)
    axes[1].set_title('Detection map')
    axes[1].axis('off')
    axes[2].imshow(ct_arr[best_z],  cmap='gray', vmin=-200, vmax=300)
    axes[2].imshow(det_arr[best_z], cmap='hot',  vmin=0,    vmax=1,  alpha=0.5)
    axes[2].set_title(f'Overlay  |  score = {likelihoods[case_id]:.4f}')
    axes[2].axis('off')
    fig.suptitle(case_id, fontsize=12)
    plt.tight_layout()
    plt.show()

## 14. (Optional) Stage-2 training from scratch

Skip if using the pretrained Stage-2 checkpoints.

**Requirements before running:**
- Section 8 (plan & preprocess) must have completed successfully
- GPU with ≥ 24 GB VRAM recommended

In [ ]:
TRAIN_STAGE2 = False   # set to True to enable

if TRAIN_STAGE2:
    os.environ['nnUNet_results']      = MODEL_DIR
    os.environ['nnUNet_raw']          = NNUNET_RAW
    os.environ['nnUNet_preprocessed'] = NNUNET_PREPROCESSED

    for fold in range(N_FOLDS):
        train_cmd = [
            'nnUNetv2_train',
            str(STAGE2_TASK),
            '3d_fullres',
            str(fold),
            '-tr', STAGE2_TRAINER,
            '-p',  STAGE2_PLAN,
        ]
        print(f'Training fold {fold}:', ' '.join(train_cmd))
        subprocess.check_call(train_cmd)
else:
    print('Training skipped (TRAIN_STAGE2=False). Using pretrained checkpoints.')

## 15. Summary report

In [ ]:
with open(osp.join(OUTPUT_DIR, 'pdac-likelihood.json')) as f:
    scores = json.load(f)

print('=' * 58)
print('  PanDx — patient-level PDAC likelihood scores')
print('=' * 58)
for case, s in sorted(scores.items()):
    bar   = '█' * int(s * 40)
    label = 'HIGH' if s >= 0.5 else 'low '
    print(f'  {case:35s}  {s:.4f}  {label}  {bar}')
print('=' * 58)

vals = list(scores.values())
if vals:
    plt.figure(figsize=(max(4, len(vals) * 1.5), 4))
    plt.bar(scores.keys(), vals,
            color=['tomato' if v >= 0.5 else 'steelblue' for v in vals])
    plt.axhline(0.5, color='red', linestyle='--', label='threshold = 0.5')
    plt.ylim(0, 1)
    plt.ylabel('PDAC likelihood')
    plt.title('Patient-level detection scores')
    plt.xticks(rotation=30, ha='right')
    plt.legend()
    plt.tight_layout()
    plt.show()

## 16. Cleanup intermediate files

Removes the `itm/` working directory (low-res images, cropped images, raw .npz files).  
Final detection maps and likelihood JSON in `output/` are **preserved**.

In [ ]:
CLEANUP = False   # set to True to free disk space

if CLEANUP and osp.exists(WORKING_DIR):
    shutil.rmtree(WORKING_DIR)
    print(f'Removed: {WORKING_DIR}')
else:
    print(f'Cleanup skipped. Intermediate files at: {WORKING_DIR}')

---
# Part C — Evaluation & Verification
---

## 17. System & hardware logging

Document GPU model, memory, and timing for reproducibility reporting.

In [ ]:
import platform, datetime
import torch

print('=' * 60)
print('  Hardware & Environment Log')
print('=' * 60)
print(f'  Date/Time  : {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'  Python     : {platform.python_version()}')
print(f'  PyTorch    : {torch.__version__}')
print(f'  Platform   : {platform.platform()}')

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'  GPU name   : {gpu.name}')
    print(f'  GPU memory : {gpu.total_memory / 1e9:.2f} GB')
    print(f'  CUDA ver   : {torch.version.cuda}')
    print(f'  GPU count  : {torch.cuda.device_count()}')
else:
    print('  GPU        : NOT available — running on CPU')
print('=' * 60)

hw_log = {
    'date':           datetime.datetime.now().isoformat(),
    'python':         platform.python_version(),
    'pytorch':        torch.__version__,
    'platform':       platform.platform(),
    'gpu_name':       torch.cuda.get_device_properties(0).name if torch.cuda.is_available() else 'CPU',
    'gpu_memory_gb':  round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2) if torch.cuda.is_available() else 0,
    'cuda_version':   torch.version.cuda if torch.cuda.is_available() else 'N/A',
}
write_json(osp.join(OUTPUT_DIR, 'hardware_log.json'), hw_log)
print(f'Log saved to: {osp.join(OUTPUT_DIR, "hardware_log.json")}')

## 18. Build ground-truth labels for evaluation

Match each case's predicted likelihood score to its ground-truth PDAC label (1 = PDAC+, 0 = PDAC-).

In [ ]:
# Load the predicted likelihood scores
with open(osp.join(OUTPUT_DIR, 'pdac-likelihood.json')) as f:
    likelihoods = json.load(f)

# Load ground truth from the DASE split CSV
split_csv = osp.join(REPO_ROOT, 'workspace', 'dase_split.csv')

if osp.exists(split_csv):
    gt_df = pd.read_csv(split_csv)[['case_id', 'is_pdac']]
else:
    # Build ground truth directly from label masks if CSV not available
    gt_records = []
    for case_id in likelihoods:
        lbl_path = None
        for ext in ['.nii.gz', '.mha', '.nii']:
            p = osp.join(LABEL_DIR, f'{case_id}{ext}')
            if osp.exists(p):
                lbl_path = p
                break
        if lbl_path:
            arr = sitk.GetArrayFromImage(sitk.ReadImage(lbl_path))
            gt_records.append({'case_id': case_id, 'is_pdac': bool((arr == 2).any())})
    gt_df = pd.DataFrame(gt_records)

# Build aligned arrays: predicted score vs ground truth label
eval_df = pd.DataFrame({
    'case_id':    list(likelihoods.keys()),
    'pred_score': list(likelihoods.values()),
}).merge(gt_df, on='case_id', how='inner')

y_true  = eval_df['is_pdac'].astype(int).values
y_score = eval_df['pred_score'].values

print(f'Cases for evaluation : {len(eval_df)}')
print(f'PDAC positive        : {y_true.sum()}')
print(f'PDAC negative        : {(y_true == 0).sum()}')
print(f'Score range          : [{y_score.min():.4f}, {y_score.max():.4f}]')

## 19. Key metric reproduction

**Targets from Liu et al. (2025):**
| Metric | PanDx (paper) | Challenge baseline |
|---|---|---|
| AUROC | **0.9263** | 0.9223 |
| Average Precision | **0.7243** | 0.6335 |

In [ ]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve
)

# ── Compute metrics ───────────────────────────────────────────────────────────
auroc = roc_auc_score(y_true, y_score)
ap    = average_precision_score(y_true, y_score)

TARGET_AUROC = 0.9263
TARGET_AP    = 0.7243
BASE_AUROC   = 0.9223
BASE_AP      = 0.6335

print('=' * 58)
print('  Metric Reproduction Results')
print('=' * 58)
print(f'  {"Metric":<25} {"Ours":>8} {"Target":>8} {"Baseline":>10} {"Gap":>8}')
print(f'  {"-"*55}')
print(f'  {"AUROC":<25} {auroc:>8.4f} {TARGET_AUROC:>8.4f} {BASE_AUROC:>10.4f} {auroc-TARGET_AUROC:>+8.4f}')
print(f'  {"Average Precision":<25} {ap:>8.4f} {TARGET_AP:>8.4f} {BASE_AP:>10.4f} {ap-TARGET_AP:>+8.4f}')
print('=' * 58)

print()
print('  AUROC: ' + ('REPRODUCED ✅ (within 0.01)' if abs(auroc - TARGET_AUROC) < 0.01 else f'DEVIATION ⚠️  (diff = {auroc-TARGET_AUROC:+.4f})'))
print('  AP   : ' + ('REPRODUCED ✅ (within 0.02)' if abs(ap - TARGET_AP) < 0.02 else f'DEVIATION ⚠️  (diff = {ap-TARGET_AP:+.4f})'))
print('  vs Baseline AUROC: ' + (f'BETTER by {auroc-BASE_AUROC:+.4f} ✅' if auroc > BASE_AUROC else f'WORSE by {auroc-BASE_AUROC:+.4f} ❌'))
print('  vs Baseline AP   : ' + (f'BETTER by {ap-BASE_AP:+.4f} ✅'    if ap > BASE_AP    else f'WORSE by {ap-BASE_AP:+.4f} ❌'))

write_json(osp.join(OUTPUT_DIR, 'evaluation_metrics.json'), {
    'AUROC': round(auroc, 4), 'Average_Precision': round(ap, 4),
    'target_AUROC': TARGET_AUROC, 'target_AP': TARGET_AP,
    'baseline_AUROC': BASE_AUROC, 'baseline_AP': BASE_AP,
    'n_cases': int(len(eval_df)), 'n_pdac_pos': int(y_true.sum()),
})
print(f'\nMetrics saved to: {osp.join(OUTPUT_DIR, "evaluation_metrics.json")}')

In [ ]:
# ── ROC and Precision-Recall curves ──────────────────────────────────────────
fpr, tpr, _   = roc_curve(y_true, y_score)
prec, rec, _  = precision_recall_curve(y_true, y_score)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(fpr, tpr, color='tomato', lw=2, label=f'PanDx (AUROC = {auroc:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title(f'ROC Curve\nTarget AUROC = {TARGET_AUROC} | Baseline = {BASE_AUROC}')
axes[0].legend(fontsize=9)
axes[0].set_xlim([0, 1]); axes[0].set_ylim([0, 1])

axes[1].plot(rec, prec, color='steelblue', lw=2, label=f'PanDx (AP = {ap:.4f})')
axes[1].axhline(y_true.mean(), color='k', linestyle='--', lw=1,
                label=f'Random (prevalence = {y_true.mean():.2f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title(f'Precision-Recall Curve\nTarget AP = {TARGET_AP} | Baseline = {BASE_AP}')
axes[1].legend(fontsize=9)
axes[1].set_xlim([0, 1]); axes[1].set_ylim([0, 1])

plt.tight_layout()
plt.savefig(osp.join(OUTPUT_DIR, 'roc_pr_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {osp.join(OUTPUT_DIR, "roc_pr_curves.png")}')

## 20. Ablation — DASE vs no-DASE split

**Paper claim:** DASE improves AP from 0.7983 → 0.8247 on validation set. AUROC stays stable.

We verify by comparing per-fold metrics under DASE vs standard random stratified split.

In [ ]:
from sklearn.model_selection import StratifiedKFold

# ── Build no-DASE (standard stratified) split ─────────────────────────────────
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
eval_df['fold_nodase'] = -1
for fold_idx, (_, val_idx) in enumerate(skf.split(eval_df, eval_df['is_pdac'])):
    eval_df.loc[eval_df.index[val_idx], 'fold_nodase'] = fold_idx

# Load DASE fold assignments
if osp.exists(split_csv):
    dase_folds = pd.read_csv(split_csv)[['case_id', 'fold']].rename(columns={'fold': 'fold_dase'})
    eval_df = eval_df.merge(dase_folds, on='case_id', how='left')
else:
    eval_df['fold_dase'] = eval_df['fold_nodase']

# ── Per-fold metrics for both splits ─────────────────────────────────────────
results = {'dase': [], 'nodase': []}
for split_name, fold_col in [('dase', 'fold_dase'), ('nodase', 'fold_nodase')]:
    for fold in range(N_FOLDS):
        val = eval_df[eval_df[fold_col] == fold]
        if val['is_pdac'].sum() == 0 or val['is_pdac'].sum() == len(val):
            continue
        results[split_name].append({
            'fold':  fold,
            'auroc': roc_auc_score(val['is_pdac'], val['pred_score']),
            'ap':    average_precision_score(val['is_pdac'], val['pred_score']),
        })

dase_r   = pd.DataFrame(results['dase'])
nodase_r = pd.DataFrame(results['nodase'])

print('=' * 62)
print('  DASE vs No-DASE — per-fold metrics')
print('=' * 62)
print(f'  {"":8} {"DASE AUROC":>12} {"DASE AP":>10} {"Rand AUROC":>12} {"Rand AP":>10}')
print(f'  {"-"*54}')
for f in range(N_FOLDS):
    d = dase_r[dase_r.fold == f]
    n = nodase_r[nodase_r.fold == f]
    if len(d) and len(n):
        print(f'  Fold {f}   {d["auroc"].values[0]:>12.4f} {d["ap"].values[0]:>10.4f} '
              f'{n["auroc"].values[0]:>12.4f} {n["ap"].values[0]:>10.4f}')
print(f'  {"-"*54}')
print(f'  {"Mean":<8} {dase_r["auroc"].mean():>12.4f} {dase_r["ap"].mean():>10.4f} '
      f'{nodase_r["auroc"].mean():>12.4f} {nodase_r["ap"].mean():>10.4f}')
print(f'  {"Std":<8} {dase_r["auroc"].std():>12.4f} {dase_r["ap"].std():>10.4f} '
      f'{nodase_r["auroc"].std():>12.4f} {nodase_r["ap"].std():>10.4f}')
print('=' * 62)

ap_diff = dase_r['ap'].mean() - nodase_r['ap'].mean()
print(f'\n  AP improvement from DASE : {ap_diff:+.4f}')
print(f'  Paper reports            : +0.0264 (0.7983 → 0.8247)')
print('  DASE AP improvement ' + ('CONFIRMED ✅' if ap_diff > 0 else 'NOT confirmed on this subset ⚠️'))

In [ ]:
# ── Visualise DASE vs no-DASE per-fold ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(N_FOLDS);  w = 0.35

for ax, metric, title, target in [
    (axes[0], 'auroc', 'AUROC per fold — DASE vs No-DASE', TARGET_AUROC),
    (axes[1], 'ap',    'AP per fold — DASE vs No-DASE',    TARGET_AP),
]:
    dv = [dase_r[dase_r.fold == f][metric].values[0]   if len(dase_r[dase_r.fold == f])   else 0 for f in range(N_FOLDS)]
    nv = [nodase_r[nodase_r.fold == f][metric].values[0] if len(nodase_r[nodase_r.fold == f]) else 0 for f in range(N_FOLDS)]
    ax.bar(x - w/2, dv, w, label='DASE',    color='tomato',    edgecolor='white')
    ax.bar(x + w/2, nv, w, label='No-DASE', color='steelblue', edgecolor='white')
    ax.axhline(target, color='black', linestyle='--', lw=1.5, label=f'Paper target ({target})')
    ax.set_xticks(x); ax.set_xticklabels([f'Fold {i}' for i in range(N_FOLDS)])
    ax.set_ylabel(metric.upper()); ax.set_title(title)
    ax.legend(fontsize=8); ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(osp.join(OUTPUT_DIR, 'dase_ablation.png'), dpi=150, bbox_inches='tight')
plt.show()

## 21. α sensitivity curve (reproduces Fig. 2 of the paper)

Sweep `1/α` from 1 to 30. The paper reports **1/α = 15 is optimal**.
We verify by computing AUROC and AP at each value using the existing probability maps.

In [ ]:
alpha_values = list(range(1, 31))
alpha_aurocs = []
alpha_aps    = []

npz_fps_eval = sorted(glob(osp.join(CROPPED_PRED_DIR, '*.npz')))

print('Sweeping 1/α from 1 to 30...')
for inv_a in tqdm(alpha_values, desc='α sweep'):
    scores_a = {}
    for npz_fp, img_fp in zip(npz_fps_eval, infer_img_paths):
        ext     = get_file_extension(img_fp)
        case_id = osp.basename(npz_fp)[:-4]
        ref_img = sitk.ReadImage(img_fp, sitk.sitkFloat32)
        prob_map = postprocess_probmap(npz_fp)
        _, score = build_full_size_detection_map(
            prob_map, crop_coordinates[case_id], ref_img, inv_alpha=inv_a
        )
        scores_a[case_id] = score

    a_eval = eval_df.copy()
    a_eval['pred_score'] = a_eval['case_id'].map(scores_a)
    a_eval = a_eval.dropna(subset=['pred_score'])

    if a_eval['is_pdac'].sum() > 0:
        alpha_aurocs.append(roc_auc_score(a_eval['is_pdac'], a_eval['pred_score']))
        alpha_aps.append(average_precision_score(a_eval['is_pdac'], a_eval['pred_score']))
    else:
        alpha_aurocs.append(np.nan)
        alpha_aps.append(np.nan)

best_auroc_idx = int(np.nanargmax(alpha_aurocs))
best_ap_idx    = int(np.nanargmax(alpha_aps))
print(f'\nOptimal 1/α for AUROC : {alpha_values[best_auroc_idx]}  (AUROC = {alpha_aurocs[best_auroc_idx]:.4f})')
print(f'Optimal 1/α for AP    : {alpha_values[best_ap_idx]}  (AP    = {alpha_aps[best_ap_idx]:.4f})')
print(f'Paper reports optimal : 1/α = 15')
print('α = 15 is optimal: ' + ('CONFIRMED ✅' if alpha_values[best_ap_idx] == 15 else f'Best found at {alpha_values[best_ap_idx]} ⚠️'))

In [ ]:
# ── Plot α sensitivity curve (Fig. 2 reproduction) ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, vals, metric_name, best_idx, target in [
    (axes[0], alpha_aurocs, 'AUROC', best_auroc_idx, TARGET_AUROC),
    (axes[1], alpha_aps,    'AP',    best_ap_idx,    TARGET_AP),
]:
    ax.plot(alpha_values, vals, 'o-', color='steelblue', lw=2, markersize=5)
    ax.axvline(15, color='red',   linestyle='--', lw=2,   label='Paper optimal (1/α = 15)')
    ax.axvline(alpha_values[best_idx], color='green', linestyle=':',
               lw=1.5, label=f'Our optimal (1/α = {alpha_values[best_idx]})')
    ax.axhline(target, color='gray', linestyle='--', lw=1, label=f'Paper target ({target})')
    ax.set_xlabel('1/α  (inverse alpha)')
    ax.set_ylabel(metric_name)
    ax.set_title(f'{metric_name} vs 1/α  —  Fig. 2 reproduction')
    ax.legend(fontsize=8)
    ax.set_xlim([1, 30])

plt.tight_layout()
plt.savefig(osp.join(OUTPUT_DIR, 'alpha_sensitivity.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {osp.join(OUTPUT_DIR, "alpha_sensitivity.png")}')

## 22. Inference timing & GPU memory usage

In [ ]:
import torch

timing_records = []
for npz_fp, img_fp in zip(npz_fps_eval, infer_img_paths):
    ext     = get_file_extension(img_fp)
    case_id = osp.basename(npz_fp)[:-4]

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    t0 = time.time()
    ref_img  = sitk.ReadImage(img_fp, sitk.sitkFloat32)
    prob_map = postprocess_probmap(npz_fp)
    _, score = build_full_size_detection_map(
        prob_map, crop_coordinates[case_id], ref_img, INV_ALPHA
    )
    elapsed = time.time() - t0

    peak_mem = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
    timing_records.append({
        'case_id':         case_id,
        'postproc_time_s': round(elapsed, 3),
        'peak_gpu_mem_gb': round(peak_mem, 3),
        'likelihood':      round(score, 4),
    })

timing_df = pd.DataFrame(timing_records)
print('Post-processing timing per case:')
print(timing_df.to_string(index=False))
print(f'\nMean post-processing time : {timing_df["postproc_time_s"].mean():.3f}s per case')
print(f'Peak GPU memory           : {timing_df["peak_gpu_mem_gb"].max():.3f} GB')

timing_df.to_csv(osp.join(OUTPUT_DIR, 'timing_log.csv'), index=False)
print(f'Timing log saved to: {osp.join(OUTPUT_DIR, "timing_log.csv")}')

## 23. Final documentation report

Consolidates all results into one report with honest documentation of uncertainty and limitations.

In [ ]:
import datetime

print('=' * 70)
print('  PanDx Reproduction — Final Report')
print(f'  Generated: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('=' * 70)

print('\n── 1. Hardware ──────────────────────────────────────────────────────')
with open(osp.join(OUTPUT_DIR, 'hardware_log.json')) as f:
    hw = json.load(f)
print(f'  GPU       : {hw["gpu_name"]}')
print(f'  GPU memory: {hw["gpu_memory_gb"]} GB')
print(f'  CUDA      : {hw["cuda_version"]}')
print(f'  PyTorch   : {hw["pytorch"]}')

print('\n── 2. Key Metrics ───────────────────────────────────────────────────')
auroc_match = '✅' if abs(auroc - TARGET_AUROC) < 0.01 else '⚠️ '
ap_match    = '✅' if abs(ap - TARGET_AP) < 0.02 else '⚠️ '
print(f'  {"Metric":<25} {"Reproduced":>12} {"Paper":>10} {"Baseline":>10} {"Match?":>7}')
print(f'  {"-"*64}')
print(f'  {"AUROC":<25} {auroc:>12.4f} {TARGET_AUROC:>10.4f} {BASE_AUROC:>10.4f} {auroc_match:>7}')
print(f'  {"Average Precision":<25} {ap:>12.4f} {TARGET_AP:>10.4f} {BASE_AP:>10.4f} {ap_match:>7}')

print('\n── 3. DASE Ablation ─────────────────────────────────────────────────')
print(f'  Mean AP with DASE    : {dase_r["ap"].mean():.4f} ± {dase_r["ap"].std():.4f}')
print(f'  Mean AP without DASE : {nodase_r["ap"].mean():.4f} ± {nodase_r["ap"].std():.4f}')
print(f'  AP improvement       : {dase_r["ap"].mean() - nodase_r["ap"].mean():+.4f}  (paper: +0.0264)')

print('\n── 4. α Sensitivity ─────────────────────────────────────────────────')
print(f'  Optimal 1/α (AUROC)  : {alpha_values[best_auroc_idx]}')
print(f'  Optimal 1/α (AP)     : {alpha_values[best_ap_idx]}')
print(f'  Paper reports optimal: 15')

print('\n── 5. Timing ────────────────────────────────────────────────────────')
print(f'  Mean post-processing : {timing_df["postproc_time_s"].mean():.3f}s per case')
print(f'  Peak GPU memory      : {timing_df["peak_gpu_mem_gb"].max():.3f} GB')

print('\n── 6. Limitations & Uncertainty (medical AI honesty) ────────────────')
print('  - Metrics on available subset; full PANORAMA test set may differ')
print('  - nnU-Net TTA introduces minor stochasticity across runs')
print('  - DASE ablation reuses same predictions; only split assignment varies')
print('  - α sweep uses post-processing only; full model retrain per α infeasible')
print('  - Hardware differences may cause small numerical differences vs paper')
print('  - Results must NOT be used for clinical diagnostic decisions')
print('=' * 70)

# Save full JSON report
final_report = {
    'hardware': hw,
    'metrics': {
        'AUROC': round(auroc, 4), 'AP': round(ap, 4),
        'target_AUROC': TARGET_AUROC, 'target_AP': TARGET_AP,
        'baseline_AUROC': BASE_AUROC, 'baseline_AP': BASE_AP,
    },
    'dase_ablation': {
        'mean_AP_dase':   round(dase_r['ap'].mean(), 4),
        'mean_AP_nodase': round(nodase_r['ap'].mean(), 4),
        'ap_improvement': round(dase_r['ap'].mean() - nodase_r['ap'].mean(), 4),
    },
    'alpha_sensitivity': {
        'optimal_inv_alpha_auroc': alpha_values[best_auroc_idx],
        'optimal_inv_alpha_ap':    alpha_values[best_ap_idx],
        'paper_optimal': 15,
    },
    'timing': {
        'mean_postproc_s':  round(timing_df['postproc_time_s'].mean(), 3),
        'peak_gpu_mem_gb':  round(timing_df['peak_gpu_mem_gb'].max(), 3),
    },
}
write_json(osp.join(OUTPUT_DIR, 'final_report.json'), final_report)
print(f'\nFull report saved to: {osp.join(OUTPUT_DIR, "final_report.json")}')

---
# Part D — Phase 3 Extensions
---

## 24. Checkpoint save
Save all Phase 1/2 variables so Phase 3 can reload them if the session restarts.

In [ ]:
import pickle

checkpoint = {
    'crop_coordinates':   crop_coordinates,
    'eval_df':            eval_df,
    'likelihoods':        likelihoods,
    'dase_r':             dase_r,
    'nodase_r':           nodase_r,
    'y_true':             y_true,
    'y_score':            y_score,
    'auroc':              auroc,
    'ap':                 ap,
    'infer_img_paths':    infer_img_paths,
    'PER_FOLD_PRED_DIRS': PER_FOLD_PRED_DIRS,
    'CROPPED_PRED_DIR':   CROPPED_PRED_DIR,
    'alpha_aurocs':       alpha_aurocs,
    'alpha_aps':          alpha_aps,
}

ckpt_path = osp.join(OUTPUT_DIR, 'phase2_checkpoint.pkl')
with open(ckpt_path, 'wb') as f:
    pickle.dump(checkpoint, f)
print(f'Checkpoint saved to: {ckpt_path}')

# ── If session restarted, reload from here ────────────────────────────────────
# with open(ckpt_path, 'rb') as f:
#     ck = pickle.load(f)
# crop_coordinates   = ck['crop_coordinates']
# eval_df            = ck['eval_df']
# likelihoods        = ck['likelihoods']
# dase_r             = ck['dase_r']
# nodase_r           = ck['nodase_r']
# y_true             = ck['y_true']
# y_score            = ck['y_score']
# auroc              = ck['auroc']
# ap                 = ck['ap']
# infer_img_paths    = ck['infer_img_paths']
# PER_FOLD_PRED_DIRS = ck['PER_FOLD_PRED_DIRS']
# CROPPED_PRED_DIR   = ck['CROPPED_PRED_DIR']

---
## Extension 1 — Failure Case Analysis

### 25. Identify false positives and false negatives

Using the operating-point threshold from the ROC curve (maximises Youden's J = sensitivity + specificity - 1).

In [ ]:
from sklearn.metrics import roc_curve

fpr_c, tpr_c, thresholds_c = roc_curve(y_true, y_score)

# Youden's J — operating point that maximises sensitivity + specificity
youden_idx = int(np.argmax(tpr_c - fpr_c))
OPT_THRESHOLD = float(thresholds_c[youden_idx])
print(f'Optimal threshold (Youden J): {OPT_THRESHOLD:.4f}')
print(f'  Sensitivity : {tpr_c[youden_idx]:.4f}')
print(f'  Specificity : {1 - fpr_c[youden_idx]:.4f}')

# Classify each case
eval_df['pred_label'] = (eval_df['pred_score'] >= OPT_THRESHOLD).astype(int)
eval_df['correct']    = (eval_df['pred_label'] == eval_df['is_pdac'].astype(int))

fn_df = eval_df[(eval_df['is_pdac'] == True)  & (eval_df['pred_label'] == 0)].copy()
fp_df = eval_df[(eval_df['is_pdac'] == False) & (eval_df['pred_label'] == 1)].copy()
tp_df = eval_df[(eval_df['is_pdac'] == True)  & (eval_df['pred_label'] == 1)].copy()
tn_df = eval_df[(eval_df['is_pdac'] == False) & (eval_df['pred_label'] == 0)].copy()

print(f'\nConfusion matrix at threshold {OPT_THRESHOLD:.3f}:')
print(f'  TP = {len(tp_df)}  FP = {len(fp_df)}')
print(f'  FN = {len(fn_df)}  TN = {len(tn_df)}')
print(f'\nFalse Negatives (missed PDAC): {len(fn_df)} cases')
print(f'False Positives (false alarm) : {len(fp_df)} cases')

### 26. Categorise errors into paper taxonomy + secondary sign quantification

**False negative types (from Fig. 3 of paper):**
- Isoattenuating lesion — lesion HU ≈ surrounding parenchyma HU (|diff| < 20 HU)
- Eccentric location — lesion centroid in pancreas head or tail (not body)
- Post-treatment anatomy — biliary stent present (hyperdense tubular structure in bile duct region)

**False positive types:**
- Cystic lesion — detected region has mean HU < 20 (fluid density)
- Extra-pancreatic mass — detected region centroid outside pancreas label
- Borderline heterogeneous — detected region HU std > 40

**Secondary signs checked for all errors:**
- Ductal dilation — duct label (3) volume > dataset 75th percentile
- Gland atrophy — pancreas label (1) volume < dataset 25th percentile

In [ ]:
def get_label_volumes(case_id):
    """Return voxel counts for each label class in a case's mask."""
    vols = {}
    for ext in ['.nii.gz', '.mha', '.nii']:
        p = osp.join(LABEL_DIR, f'{case_id}{ext}')
        if osp.exists(p):
            mask  = sitk.ReadImage(p)
            arr   = sitk.GetArrayFromImage(mask)
            sp    = mask.GetSpacing()
            vv    = sp[0] * sp[1] * sp[2]
            for lbl in range(7):
                vols[lbl] = int((arr == lbl).sum()) * vv
            return vols, arr, mask
    return {i: 0 for i in range(7)}, None, None


def get_region_hu_stats(ct_arr, det_arr, threshold=0.1):
    """HU stats of voxels above detection threshold."""
    region = ct_arr[det_arr > threshold]
    if len(region) == 0:
        return {'mean': np.nan, 'std': np.nan}
    return {'mean': float(region.mean()), 'std': float(region.std())}


# ── Compute dataset-wide secondary sign thresholds ───────────────────────────
duct_vols, pancreas_vols = [], []
for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc='Secondary sign thresholds'):
    vols, _, _ = get_label_volumes(row['case_id'])
    duct_vols.append(vols.get(3, 0))
    pancreas_vols.append(vols.get(1, 0))

duct_75th    = float(np.percentile(duct_vols, 75))
pancreas_25th = float(np.percentile(pancreas_vols, 25))
print(f'Duct dilation threshold (75th pct)  : {duct_75th:.0f} mm³')
print(f'Gland atrophy threshold (25th pct)  : {pancreas_25th:.0f} mm³')

# ── Analyse false negatives ───────────────────────────────────────────────────
fn_records = []
for _, row in fn_df.iterrows():
    cid   = row['case_id']
    vols, mask_arr, mask_itk = get_label_volumes(cid)
    ct_itk = sitk.ReadImage(osp.join(INPUT_DIR, f'{cid}.mha'), sitk.sitkFloat32) \
             if osp.exists(osp.join(INPUT_DIR, f'{cid}.mha')) else None

    # Lesion vs parenchyma HU difference
    iso = False
    if ct_itk is not None and mask_arr is not None:
        ct_arr = sitk.GetArrayFromImage(ct_itk)
        lesion_hu = ct_arr[mask_arr == 2].mean() if (mask_arr == 2).any() else np.nan
        parench_hu = ct_arr[mask_arr == 1].mean() if (mask_arr == 1).any() else np.nan
        iso = abs(lesion_hu - parench_hu) < 20 if not np.isnan(lesion_hu) else False

    # Eccentric location — centroid in outer 33% of pancreas x-axis
    eccentric = False
    if mask_arr is not None and (mask_arr == 2).any():
        nz    = np.nonzero(mask_arr == 2)
        pan_nz = np.nonzero(mask_arr == 1)
        if len(pan_nz[2]) > 0:
            pan_x_range = pan_nz[2].max() - pan_nz[2].min()
            lesion_x    = nz[2].mean() - pan_nz[2].min()
            rel_pos     = lesion_x / (pan_x_range + 1e-6)
            eccentric   = rel_pos < 0.33 or rel_pos > 0.67

    # Post-treatment: hyperdense structure in bile duct region (label 4 present + high HU)
    stent = vols.get(4, 0) > 0

    # Secondary signs
    ductal_dilation = vols.get(3, 0) > duct_75th
    gland_atrophy   = vols.get(1, 0) < pancreas_25th

    fn_records.append({
        'case_id': cid, 'score': row['pred_score'],
        'isoattenuating': iso, 'eccentric': eccentric, 'post_treatment': stent,
        'ductal_dilation': ductal_dilation, 'gland_atrophy': gland_atrophy,
        'any_secondary_sign': ductal_dilation or gland_atrophy,
    })

fn_analysis = pd.DataFrame(fn_records)

# ── Analyse false positives ───────────────────────────────────────────────────
fp_records = []
for _, row in fp_df.iterrows():
    cid  = row['case_id']
    vols, mask_arr, mask_itk = get_label_volumes(cid)
    det_path = osp.join(OUTPUT_DIR, 'pdac-detection-map', f'{cid}.nii.gz')
    det_arr  = sitk.GetArrayFromImage(sitk.ReadImage(det_path)) if osp.exists(det_path) else None

    ct_itk = sitk.ReadImage(osp.join(INPUT_DIR, f'{cid}.mha'), sitk.sitkFloat32) \
             if osp.exists(osp.join(INPUT_DIR, f'{cid}.mha')) else None

    cystic = False; extrapancreatic = False; heterogeneous = False
    if ct_itk is not None and det_arr is not None:
        ct_arr   = sitk.GetArrayFromImage(ct_itk)
        hu_stats = get_region_hu_stats(ct_arr, det_arr)
        cystic        = hu_stats['mean'] < 20
        heterogeneous = hu_stats['std'] > 40
        if mask_arr is not None and (mask_arr == 1).any():
            det_nz    = np.nonzero(det_arr > 0.1)
            if len(det_nz[0]) > 0:
                det_centroid = np.array([det_nz[i].mean() for i in range(3)])
                pan_nz       = np.nonzero(mask_arr == 1)
                pan_min = np.array([pan_nz[i].min() for i in range(3)])
                pan_max = np.array([pan_nz[i].max() for i in range(3)])
                extrapancreatic = bool(np.any(det_centroid < pan_min) or np.any(det_centroid > pan_max))

    ductal_dilation = vols.get(3, 0) > duct_75th
    gland_atrophy   = vols.get(1, 0) < pancreas_25th

    fp_records.append({
        'case_id': cid, 'score': row['pred_score'],
        'cystic': cystic, 'extrapancreatic': extrapancreatic, 'heterogeneous': heterogeneous,
        'ductal_dilation': ductal_dilation, 'gland_atrophy': gland_atrophy,
        'any_secondary_sign': ductal_dilation or gland_atrophy,
    })

fp_analysis = pd.DataFrame(fp_records)

# ── Print summary ─────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('  False Negative Analysis')
print('=' * 60)
if len(fn_analysis):
    for col, label in [('isoattenuating','Isoattenuating'), ('eccentric','Eccentric location'),
                       ('post_treatment','Post-treatment anatomy'), ('any_secondary_sign','Any secondary sign')]:
        n = fn_analysis[col].sum()
        pct = 100 * n / len(fn_analysis)
        print(f'  {label:<30}: {n}/{len(fn_analysis)}  ({pct:.1f}%)')
else:
    print('  No false negatives found.')

print('\n' + '=' * 60)
print('  False Positive Analysis')
print('=' * 60)
if len(fp_analysis):
    for col, label in [('cystic','Cystic lesion'), ('extrapancreatic','Extra-pancreatic mass'),
                       ('heterogeneous','Heterogeneous region'), ('any_secondary_sign','Any secondary sign')]:
        n = fp_analysis[col].sum()
        pct = 100 * n / len(fp_analysis)
        print(f'  {label:<30}: {n}/{len(fp_analysis)}  ({pct:.1f}%)')
else:
    print('  No false positives found.')

fn_analysis.to_csv(osp.join(OUTPUT_DIR, 'fn_analysis.csv'), index=False)
fp_analysis.to_csv(osp.join(OUTPUT_DIR, 'fp_analysis.csv'), index=False)

---
## Extension 2 — Robustness Testing

### 27. Degradation helpers + Gaussian noise test

In [ ]:
def apply_gaussian_noise(itk_image, sigma_hu):
    """Add Gaussian noise (σ in HU) to a CT image."""
    arr  = sitk.GetArrayFromImage(itk_image).astype(np.float32)
    noise = np.random.default_rng(RANDOM_SEED).normal(0, sigma_hu, arr.shape).astype(np.float32)
    noisy = sitk.GetImageFromArray(arr + noise)
    noisy.CopyInformation(itk_image)
    return noisy


def apply_resolution_degradation(itk_image):
    """Downsample by 2x then upsample back — simulates thick-slice scanner."""
    orig_size    = itk_image.GetSize()
    orig_spacing = itk_image.GetSpacing()
    low_spacing  = tuple(s * 2 for s in orig_spacing)
    downsampled  = resample_img(itk_image, out_spacing=low_spacing, is_label=False)
    restored     = resample_img(downsampled, out_spacing=orig_spacing,
                                is_label=False, out_size=list(orig_size))
    return restored


def apply_partial_fov(itk_image, crop_fraction=0.2):
    """Crop top and bottom slices to simulate partial field-of-view."""
    arr   = sitk.GetArrayFromImage(itk_image)
    z     = arr.shape[0]
    start = int(z * crop_fraction)
    end   = int(z * (1 - crop_fraction))
    cropped = sitk.GetImageFromArray(arr[start:end])
    # Adjust origin to account for crop
    orig   = list(itk_image.GetOrigin())
    sp     = itk_image.GetSpacing()
    orig[2] = orig[2] + start * sp[2]
    cropped.SetSpacing(itk_image.GetSpacing())
    cropped.SetDirection(itk_image.GetDirection())
    cropped.SetOrigin(tuple(orig))
    return cropped


def run_degraded_inference(degradation_name, degraded_img_dir):
    """Run full Stage1→Stage2→postprocess on a folder of degraded images."""
    wdir = osp.join(WORKING_DIR, f'robust_{degradation_name}')
    os.makedirs(wdir, exist_ok=True)

    # Stage 1
    low_dir  = osp.join(wdir, 'LowImages')
    pred_dir = osp.join(wdir, 'LowPred')
    downsample_dataset(degraded_img_dir, low_dir, DOWNSAMPLE_SPACING)
    run_nnunet_predict(MODEL_DIR, low_dir, pred_dir, STAGE1_TASK,
                       STAGE1_TRAINER, STAGE1_PLAN, save_probs=True, tta=False)

    # Crop ROI
    crop_dir   = osp.join(wdir, 'CroppedImages')
    crop_coords_deg = crop_roi(degraded_img_dir, pred_dir, crop_dir, ROI_MARGINS)

    # Stage 2
    s2_dir = osp.join(wdir, 'CroppedPred')
    run_nnunet_predict(MODEL_DIR, crop_dir, s2_dir, STAGE2_TASK,
                       STAGE2_TRAINER, STAGE2_PLAN, save_probs=True, tta=False)

    # Post-process
    from sklearn.metrics import roc_auc_score, average_precision_score
    scores_deg = {}
    for npz_fp in sorted(glob(osp.join(s2_dir, '*.npz'))):
        cid = osp.basename(npz_fp)[:-4]
        # Find matching original image for reference geometry
        ref_path = None
        for ext in ['.mha', '.nii.gz', '.nii']:
            p = osp.join(INPUT_DIR, f'{cid}{ext}')
            if osp.exists(p):
                ref_path = p; break
        if ref_path is None:
            continue
        ref_img  = sitk.ReadImage(ref_path, sitk.sitkFloat32)
        prob_map = postprocess_probmap(npz_fp)
        if cid in crop_coords_deg:
            _, score = build_full_size_detection_map(prob_map, crop_coords_deg[cid], ref_img, INV_ALPHA)
            scores_deg[cid] = score

    deg_eval = eval_df[eval_df['case_id'].isin(scores_deg)].copy()
    deg_eval['pred_score'] = deg_eval['case_id'].map(scores_deg)
    if deg_eval['is_pdac'].sum() == 0:
        return {'auroc': np.nan, 'ap': np.nan}
    return {
        'auroc': roc_auc_score(deg_eval['is_pdac'], deg_eval['pred_score']),
        'ap':    average_precision_score(deg_eval['is_pdac'], deg_eval['pred_score']),
    }


# ── Apply and test Gaussian noise ─────────────────────────────────────────────
robustness_results = {'clean': {'auroc': auroc, 'ap': ap}}

for sigma in [10, 20, 40]:
    print(f'\nApplying Gaussian noise σ={sigma} HU...')
    noise_dir = osp.join(WORKING_DIR, f'noise_sigma{sigma}')
    os.makedirs(noise_dir, exist_ok=True)
    for img_fp in tqdm(infer_img_paths, desc=f'σ={sigma}'):
        ext  = get_file_extension(img_fp)
        cid  = osp.basename(img_fp).replace(ext, '')
        img  = sitk.ReadImage(img_fp, sitk.sitkFloat32)
        noisy = apply_gaussian_noise(img, sigma)
        sitk.WriteImage(noisy, osp.join(noise_dir, osp.basename(img_fp)))
    metrics = run_degraded_inference(f'noise{sigma}', noise_dir)
    robustness_results[f'noise_sigma{sigma}'] = metrics
    print(f'  AUROC={metrics["auroc"]:.4f}  AP={metrics["ap"]:.4f}')

print('\nGaussian noise robustness done.')

### 28. Resolution degradation + partial field-of-view tests

In [ ]:
# ── Resolution degradation ────────────────────────────────────────────────────
print('Applying 2x resolution degradation...')
res_dir = osp.join(WORKING_DIR, 'resolution_degraded')
os.makedirs(res_dir, exist_ok=True)
for img_fp in tqdm(infer_img_paths, desc='Resolution'):
    img = sitk.ReadImage(img_fp, sitk.sitkFloat32)
    sitk.WriteImage(apply_resolution_degradation(img),
                    osp.join(res_dir, osp.basename(img_fp)))
metrics = run_degraded_inference('resolution', res_dir)
robustness_results['resolution_2x'] = metrics
print(f'  AUROC={metrics["auroc"]:.4f}  AP={metrics["ap"]:.4f}')

# ── Partial field of view ─────────────────────────────────────────────────────
print('\nApplying partial field-of-view (20% crop)...')
fov_dir = osp.join(WORKING_DIR, 'partial_fov')
os.makedirs(fov_dir, exist_ok=True)
for img_fp in tqdm(infer_img_paths, desc='Partial FOV'):
    img = sitk.ReadImage(img_fp, sitk.sitkFloat32)
    sitk.WriteImage(apply_partial_fov(img, 0.2),
                    osp.join(fov_dir, osp.basename(img_fp)))
metrics = run_degraded_inference('partialfov', fov_dir)
robustness_results['partial_fov'] = metrics
print(f'  AUROC={metrics["auroc"]:.4f}  AP={metrics["ap"]:.4f}')

print('\nAll robustness tests complete.')

### 29. Robustness summary plot

In [ ]:
rob_df = pd.DataFrame(robustness_results).T.reset_index()
rob_df.columns = ['condition', 'auroc', 'ap']
rob_df['auroc_delta'] = rob_df['auroc'] - auroc
rob_df['ap_delta']    = rob_df['ap']    - ap

print('=' * 62)
print('  Robustness Results — AUROC and AP delta vs clean baseline')
print('=' * 62)
print(f'  {"Condition":<22} {"AUROC":>8} {"Δ AUROC":>10} {"AP":>8} {"Δ AP":>8}')
print(f'  {"-"*58}')
for _, row in rob_df.iterrows():
    print(f'  {row["condition"]:<22} {row["auroc"]:>8.4f} {row["auroc_delta"]:>+10.4f} '
          f'{row["ap"]:>8.4f} {row["ap_delta"]:>+8.4f}')
print('=' * 62)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colours = ['#4CAF50' if v >= 0 else '#F44336' for v in rob_df['auroc_delta']]
for ax, col, title in [
    (axes[0], 'auroc_delta', 'AUROC delta vs clean baseline'),
    (axes[1], 'ap_delta',    'AP delta vs clean baseline'),
]:
    vals = rob_df[col].values
    cols = ['#4CAF50' if v >= 0 else '#F44336' for v in vals]
    ax.bar(rob_df['condition'], vals, color=cols, edgecolor='white')
    ax.axhline(0, color='black', lw=1.5)
    ax.set_ylabel('Δ metric')
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(osp.join(OUTPUT_DIR, 'robustness_results.png'), dpi=150, bbox_inches='tight')
plt.show()
rob_df.to_csv(osp.join(OUTPUT_DIR, 'robustness_results.csv'), index=False)
print(f'Saved: robustness_results.png and .csv')

---
## Extension 3 — Fairness & Subgroup Analysis

### 30. Subgroup AUROC / AP by lesion size, sex, and age

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

def subgroup_metrics(df, group_col, score_col='pred_score', label_col='is_pdac'):
    """Compute AUROC and AP for each unique value of group_col."""
    records = []
    for grp in sorted(df[group_col].dropna().unique()):
        sub = df[df[group_col] == grp]
        if sub[label_col].sum() == 0 or sub[label_col].sum() == len(sub):
            continue
        records.append({
            'group':  str(grp),
            'n':      len(sub),
            'n_pos':  int(sub[label_col].sum()),
            'auroc':  round(roc_auc_score(sub[label_col], sub[score_col]), 4),
            'ap':     round(average_precision_score(sub[label_col], sub[score_col]), 4),
        })
    return pd.DataFrame(records)


# ── Attach lesion size quartile, sex, age group ───────────────────────────────
split_df = pd.read_csv(split_csv) if osp.exists(split_csv) else df[['case_id','is_pdac','pdac_vol_mm3']]

# Lesion size quartile (DASE bins) — only for PDAC+ cases
pdac_only = split_df[split_df['is_pdac'] == True].copy()
pdac_only['size_bin'] = pd.qcut(
    pdac_only['pdac_vol_mm3'], q=4,
    labels=['Q1 (0-25%)', 'Q2 (25-50%)', 'Q3 (50-75%)', 'Q4 (75-100%)']
)
eval_df = eval_df.merge(pdac_only[['case_id', 'size_bin']], on='case_id', how='left')

# Sex and age from original df (if clinical metadata was available)
if 'age' in df.columns:
    eval_df = eval_df.merge(df[['case_id', 'age', 'sex']], on='case_id', how='left')
    eval_df['age_group'] = pd.cut(
        eval_df['age'], bins=[0, 60, 70, 120],
        labels=['< 60', '60–70', '> 70'], right=False
    )

# ── Compute subgroup metrics ──────────────────────────────────────────────────
print('=' * 55)
print('  Subgroup Analysis — Lesion Size Quartiles')
print('  (PDAC+ cases only, matched to DASE bins)')
print('=' * 55)
size_metrics = subgroup_metrics(
    eval_df[eval_df['is_pdac'] == True].assign(is_pdac=1),
    'size_bin'
)
print(size_metrics.to_string(index=False))

if 'sex' in eval_df.columns and eval_df['sex'].notna().any():
    print('\n' + '=' * 55)
    print('  Subgroup Analysis — Sex')
    print('=' * 55)
    sex_metrics = subgroup_metrics(eval_df, 'sex')
    print(sex_metrics.to_string(index=False))

if 'age_group' in eval_df.columns and eval_df['age_group'].notna().any():
    print('\n' + '=' * 55)
    print('  Subgroup Analysis — Age Group')
    print('=' * 55)
    age_metrics = subgroup_metrics(eval_df, 'age_group')
    print(age_metrics.to_string(index=False))

### 31. Does DASE close the Q1 performance gap?

In [ ]:
# Compare DASE vs no-DASE per size quartile on PDAC+ cases
bins = ['Q1 (0-25%)', 'Q2 (25-50%)', 'Q3 (50-75%)', 'Q4 (75-100%)']
fairness_records = []

for bin_label in bins:
    sub = eval_df[(eval_df['is_pdac'] == True) & (eval_df['size_bin'] == bin_label)].copy()
    sub = sub.assign(is_pdac=1)
    if len(sub) < 2:
        continue
    # DASE scores (from ensemble)
    dase_ap   = average_precision_score(sub['is_pdac'], sub['pred_score']) if sub['is_pdac'].sum() > 0 else np.nan
    # No-DASE: use nodase fold assignments to get val-set scores
    nodase_sub = sub[sub['fold_nodase'].notna()].copy() if 'fold_nodase' in sub.columns else sub.copy()
    nodase_ap  = average_precision_score(nodase_sub['is_pdac'], nodase_sub['pred_score']) \
                 if len(nodase_sub) > 1 and nodase_sub['is_pdac'].sum() > 0 else np.nan
    fairness_records.append({
        'size_bin':    bin_label,
        'n':           len(sub),
        'ap_dase':     round(dase_ap, 4),
        'ap_nodase':   round(nodase_ap, 4),
        'ap_gain':     round(dase_ap - nodase_ap, 4) if not np.isnan(nodase_ap) else np.nan,
    })

fair_df = pd.DataFrame(fairness_records)
print('=' * 62)
print('  DASE Fairness — AP gain per lesion size quartile')
print('  Key question: does DASE help Q1 (hardest) most?')
print('=' * 62)
print(fair_df.to_string(index=False))
if len(fair_df):
    q1_gain = fair_df[fair_df['size_bin'] == 'Q1 (0-25%)']['ap_gain'].values
    q4_gain = fair_df[fair_df['size_bin'] == 'Q4 (75-100%)']['ap_gain'].values
    if len(q1_gain) and len(q4_gain) and not np.isnan(q1_gain[0]) and not np.isnan(q4_gain[0]):
        print(f'\n  Q1 AP gain: {q1_gain[0]:+.4f}  |  Q4 AP gain: {q4_gain[0]:+.4f}')
        print('  DASE closes Q1 gap more than Q4: ' +
              ('YES ✅' if q1_gain[0] > q4_gain[0] else 'NO ⚠️'))

# Plot
if len(fair_df):
    x  = np.arange(len(fair_df));  w = 0.35
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(x - w/2, fair_df['ap_dase'],   w, label='DASE',    color='tomato',    edgecolor='white')
    ax.bar(x + w/2, fair_df['ap_nodase'], w, label='No-DASE', color='steelblue', edgecolor='white')
    ax.set_xticks(x); ax.set_xticklabels(fair_df['size_bin'], rotation=15)
    ax.set_ylabel('Average Precision')
    ax.set_title('DASE vs No-DASE AP by lesion size quartile\n(does DASE close the Q1 gap?)')
    ax.legend(); ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig(osp.join(OUTPUT_DIR, 'fairness_dase_q1.png'), dpi=150, bbox_inches='tight')
    plt.show()
    fair_df.to_csv(osp.join(OUTPUT_DIR, 'fairness_subgroup.csv'), index=False)

---
## Extension 4 — Uncertainty Quantification

### 32. Compute fold-variance uncertainty from per-fold .npz outputs

In [ ]:
uncertainty_records = []

case_ids = [osp.basename(p)[:-4]
            for p in sorted(glob(osp.join(CROPPED_PRED_DIR, '*.npz')))]

for case_id in tqdm(case_ids, desc='Uncertainty'):
    fold_probs = []
    for fold in range(N_FOLDS):
        npz_path = osp.join(PER_FOLD_PRED_DIRS[fold], f'{case_id}.npz')
        if osp.exists(npz_path):
            fold_probs.append(np.load(npz_path)['probabilities'][1].astype(np.float32))

    if len(fold_probs) < 2:
        uncertainty_records.append({'case_id': case_id,
                                    'mean_uncertainty': np.nan,
                                    'max_uncertainty':  np.nan,
                                    'mean_prob':        np.nan})
        continue

    stack     = np.stack(fold_probs, axis=0)          # (n_folds, Z, Y, X)
    var_map   = stack.var(axis=0)                      # voxel-wise variance
    mean_map  = stack.mean(axis=0)

    uncertainty_records.append({
        'case_id':          case_id,
        'mean_uncertainty': float(var_map.mean()),
        'max_uncertainty':  float(var_map.max()),
        'mean_prob':        float(mean_map.max()),     # patient-level score
    })

unc_df = pd.DataFrame(uncertainty_records)
unc_df = unc_df.merge(eval_df[['case_id', 'is_pdac', 'pred_score',
                                'size_bin', 'correct']], on='case_id', how='left')

print(f'Uncertainty computed for {unc_df["mean_uncertainty"].notna().sum()} cases')
print('\nMean uncertainty by PDAC status:')
print(unc_df.groupby('is_pdac')[['mean_uncertainty', 'max_uncertainty']].mean().round(6).to_string())
print('\nMean uncertainty — correct vs incorrect predictions:')
print(unc_df.groupby('correct')[['mean_uncertainty', 'max_uncertainty']].mean().round(6).to_string())

### 33. Calibration — Expected Calibration Error + reliability diagram

In [ ]:
def expected_calibration_error(y_true, y_prob, n_bins=10):
    """Compute ECE and return bin-level data for reliability diagram."""
    bins    = np.linspace(0, 1, n_bins + 1)
    ece     = 0.0
    bin_data = []
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i+1])
        if mask.sum() == 0:
            bin_data.append({'bin_mid': (bins[i]+bins[i+1])/2,
                             'mean_pred': np.nan, 'frac_pos': np.nan, 'n': 0})
            continue
        mean_pred = float(y_prob[mask].mean())
        frac_pos  = float(y_true[mask].mean())
        ece      += (mask.sum() / len(y_true)) * abs(mean_pred - frac_pos)
        bin_data.append({'bin_mid': (bins[i]+bins[i+1])/2,
                         'mean_pred': mean_pred, 'frac_pos': frac_pos, 'n': int(mask.sum())})
    return ece, pd.DataFrame(bin_data)

ece_val, cal_df = expected_calibration_error(y_true, y_score)
print(f'Expected Calibration Error (ECE): {ece_val:.4f}')
print('(0.0 = perfect calibration, higher = worse)')
print()
print('Reliability diagram data:')
print(cal_df[cal_df['n'] > 0][['bin_mid', 'mean_pred', 'frac_pos', 'n']].round(3).to_string(index=False))

# ── Plot reliability diagram ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

valid = cal_df[cal_df['n'] > 0]
axes[0].plot([0,1],[0,1], 'k--', lw=1.5, label='Perfect calibration')
axes[0].bar(valid['bin_mid'], valid['frac_pos'], width=0.09,
            color='steelblue', alpha=0.7, edgecolor='white', label='Model')
axes[0].set_xlabel('Mean predicted probability')
axes[0].set_ylabel('Fraction of PDAC+ cases')
axes[0].set_title(f'Reliability Diagram\nECE = {ece_val:.4f}')
axes[0].legend(fontsize=9)
axes[0].set_xlim([0,1]); axes[0].set_ylim([0,1])

axes[1].bar(valid['bin_mid'], valid['n'], width=0.09,
            color='tomato', edgecolor='white')
axes[1].set_xlabel('Predicted probability bin')
axes[1].set_ylabel('Number of cases')
axes[1].set_title('Prediction confidence histogram')

plt.tight_layout()
plt.savefig(osp.join(OUTPUT_DIR, 'calibration_reliability.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: calibration_reliability.png')

### 34. Hypothesis test — does uncertainty correlate with failure types?

**Paper hypothesis:** high-uncertainty predictions should correlate with isoattenuating lesions, small lesion size (Q1), and degraded scan quality.

In [ ]:
from scipy import stats

valid_unc = unc_df[unc_df['mean_uncertainty'].notna()].copy()

print('=' * 65)
print('  Uncertainty Hypothesis Tests (Mann-Whitney U)')
print('  H1: errors have higher uncertainty than correct predictions')
print('  H2: Q1 (small lesions) have higher uncertainty than Q4')
print('  H3: isoattenuating FNs have higher uncertainty than other FNs')
print('=' * 65)

# H1: incorrect vs correct
correct   = valid_unc[valid_unc['correct'] == True]['mean_uncertainty'].dropna()
incorrect = valid_unc[valid_unc['correct'] == False]['mean_uncertainty'].dropna()
if len(correct) > 1 and len(incorrect) > 1:
    stat, p = stats.mannwhitneyu(incorrect, correct, alternative='greater')
    print(f'\n  H1 — Incorrect vs Correct uncertainty:')
    print(f'    Mean uncertainty incorrect : {incorrect.mean():.6f}')
    print(f'    Mean uncertainty correct   : {correct.mean():.6f}')
    print(f'    Mann-Whitney U p-value     : {p:.4f}')
    print(f'    Result: ' + ('Confirmed ✅ (p < 0.05)' if p < 0.05 else 'Not significant ⚠️'))

# H2: Q1 vs Q4 uncertainty
if 'size_bin' in valid_unc.columns:
    q1_unc = valid_unc[valid_unc['size_bin'] == 'Q1 (0-25%)']['mean_uncertainty'].dropna()
    q4_unc = valid_unc[valid_unc['size_bin'] == 'Q4 (75-100%)']['mean_uncertainty'].dropna()
    if len(q1_unc) > 1 and len(q4_unc) > 1:
        stat, p = stats.mannwhitneyu(q1_unc, q4_unc, alternative='greater')
        print(f'\n  H2 — Q1 vs Q4 uncertainty:')
        print(f'    Mean uncertainty Q1 (small): {q1_unc.mean():.6f}')
        print(f'    Mean uncertainty Q4 (large): {q4_unc.mean():.6f}')
        print(f'    Mann-Whitney U p-value      : {p:.4f}')
        print(f'    Result: ' + ('Confirmed ✅ (p < 0.05)' if p < 0.05 else 'Not significant ⚠️'))

# H3: isoattenuating FNs vs non-isoattenuating FNs
if len(fn_analysis):
    fn_unc = valid_unc[valid_unc['case_id'].isin(fn_analysis['case_id'])].copy()
    fn_unc = fn_unc.merge(fn_analysis[['case_id', 'isoattenuating']], on='case_id', how='left')
    iso_unc    = fn_unc[fn_unc['isoattenuating'] == True]['mean_uncertainty'].dropna()
    noniso_unc = fn_unc[fn_unc['isoattenuating'] == False]['mean_uncertainty'].dropna()
    if len(iso_unc) > 1 and len(noniso_unc) > 1:
        stat, p = stats.mannwhitneyu(iso_unc, noniso_unc, alternative='greater')
        print(f'\n  H3 — Isoattenuating vs non-isoattenuating FN uncertainty:')
        print(f'    Mean uncertainty isoattenuating    : {iso_unc.mean():.6f}')
        print(f'    Mean uncertainty non-isoattenuating: {noniso_unc.mean():.6f}')
        print(f'    Mann-Whitney U p-value             : {p:.4f}')
        print(f'    Result: ' + ('Confirmed ✅ (p < 0.05)' if p < 0.05 else 'Not significant ⚠️'))
    else:
        print('\n  H3: insufficient isoattenuating FN cases for test')

print('=' * 65)

# ── Plot uncertainty distributions ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Correct vs incorrect
axes[0].hist(correct,   bins=20, alpha=0.6, color='steelblue', label='Correct',   edgecolor='white')
axes[0].hist(incorrect, bins=20, alpha=0.6, color='tomato',    label='Incorrect', edgecolor='white')
axes[0].set_xlabel('Mean fold variance (uncertainty)')
axes[0].set_ylabel('Count')
axes[0].set_title('Uncertainty: correct vs incorrect predictions')
axes[0].legend()

# By size quartile
if 'size_bin' in valid_unc.columns:
    for bin_lbl, colour in zip(bins, ['#4CAF50','#2196F3','#FF9800','#F44336']):
        sub = valid_unc[valid_unc['size_bin'] == bin_lbl]['mean_uncertainty'].dropna()
        if len(sub):
            axes[1].hist(sub, bins=15, alpha=0.5, color=colour,
                         label=bin_lbl, edgecolor='white')
axes[1].set_xlabel('Mean fold variance (uncertainty)')
axes[1].set_ylabel('Count')
axes[1].set_title('Uncertainty by lesion size quartile')
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.savefig(osp.join(OUTPUT_DIR, 'uncertainty_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

unc_df.to_csv(osp.join(OUTPUT_DIR, 'uncertainty_scores.csv'), index=False)
print(f'Saved: uncertainty_distributions.png  and  uncertainty_scores.csv')

---
## 35. Phase 3 Final Report

Consolidates all extension results into one printout and JSON file.

In [ ]:
import datetime

print('=' * 70)
print('  PanDx — Phase 3 Extension Report')
print(f'  Generated: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('=' * 70)

print('\n── Extension 1: Failure Case Analysis ──────────────────────────────')
print(f'  False Negatives : {len(fn_df)} cases')
if len(fn_analysis):
    for col, lbl in [('isoattenuating','Isoattenuating'), ('eccentric','Eccentric'),
                     ('post_treatment','Post-treatment'), ('any_secondary_sign','Has secondary sign')]:
        n = fn_analysis[col].sum()
        print(f'    {lbl:<25}: {n}/{len(fn_analysis)} ({100*n/len(fn_analysis):.0f}%)')
print(f'  False Positives : {len(fp_df)} cases')
if len(fp_analysis):
    for col, lbl in [('cystic','Cystic'), ('extrapancreatic','Extra-pancreatic'),
                     ('heterogeneous','Heterogeneous'), ('any_secondary_sign','Has secondary sign')]:
        n = fp_analysis[col].sum()
        print(f'    {lbl:<25}: {n}/{len(fp_analysis)} ({100*n/len(fp_analysis):.0f}%)')

print('\n── Extension 2: Robustness ──────────────────────────────────────────')
for _, row in rob_df.iterrows():
    flag = '✅' if abs(row['auroc_delta']) < 0.02 else '⚠️ '
    print(f'  {row["condition"]:<22} AUROC Δ={row["auroc_delta"]:+.4f}  AP Δ={row["ap_delta"]:+.4f}  {flag}')

print('\n── Extension 3: Fairness ────────────────────────────────────────────')
if len(fair_df):
    for _, row in fair_df.iterrows():
        flag = '✅' if row['ap_gain'] > 0 else '⚠️ '
        print(f'  {str(row["size_bin"]):<20} AP DASE={row["ap_dase"]:.4f}  '
              f'No-DASE={row["ap_nodase"]:.4f}  gain={row["ap_gain"]:+.4f}  {flag}')

print('\n── Extension 4: Uncertainty ─────────────────────────────────────────')
print(f'  ECE                  : {ece_val:.4f}')
print(f'  Mean unc (correct)   : {correct.mean():.6f}')
print(f'  Mean unc (incorrect) : {incorrect.mean():.6f}')

print('\n── Limitations ──────────────────────────────────────────────────────')
print('  - Failure categorisation is heuristic (HU thresholds, not radiologist-confirmed)')
print('  - Robustness tests use synthetic degradation, not real degraded scans')
print('  - Subgroup analysis may be underpowered if subgroup n is small')
print('  - Uncertainty from fold variance is a proxy; not true Bayesian uncertainty')
print('  - All findings are on available subset, not the full PANORAMA test set')
print('  - MEDICAL AI: outputs must not be used for clinical decisions')
print('=' * 70)

# Save Phase 3 JSON report
p3_report = {
    'failure_analysis': {
        'n_false_negatives': len(fn_df),
        'n_false_positives': len(fp_df),
        'fn_isoattenuating_pct':   round(100*fn_analysis['isoattenuating'].mean(), 1) if len(fn_analysis) else None,
        'fn_secondary_sign_pct':   round(100*fn_analysis['any_secondary_sign'].mean(), 1) if len(fn_analysis) else None,
        'fp_cystic_pct':           round(100*fp_analysis['cystic'].mean(), 1) if len(fp_analysis) else None,
        'fp_secondary_sign_pct':   round(100*fp_analysis['any_secondary_sign'].mean(), 1) if len(fp_analysis) else None,
    },
    'robustness': rob_df.set_index('condition')[['auroc','ap','auroc_delta','ap_delta']].to_dict(),
    'fairness': fair_df.set_index('size_bin').to_dict() if len(fair_df) else {},
    'uncertainty': {
        'ECE':                  round(ece_val, 4),
        'mean_unc_correct':     round(float(correct.mean()), 6),
        'mean_unc_incorrect':   round(float(incorrect.mean()), 6),
    },
}
write_json(osp.join(OUTPUT_DIR, 'phase3_report.json'), p3_report)
print(f'\nPhase 3 report saved to: {osp.join(OUTPUT_DIR, "phase3_report.json")}')